# Step 1a: Discover the dataset

**Purpose.** Look at the Cincinnati 311 dataset before pulling it, so the snapshot in step 1b filters on a real column name instead of a guess.

**Rule for this project:** notebooks explore, scripts produce. Nothing in this notebook writes a file that later steps depend on. Anything worth keeping gets moved into `src/`.

**Why that rule exists.** A notebook can be run out of order. Cell 12 can depend on a variable defined in a cell you later deleted, and it will still work in your session and fail for everyone else. That is fine for looking around and disqualifying for a pipeline. Keeping the two separate is also the answer to an interview question you will get: "how do you know this is reproducible?"

In [ ]:
# Make the project's src/ package importable from a notebook.
#
# WHY THIS IS NEEDED: Jupyter sets the working directory to wherever the
# notebook lives (notebooks/), not the project root. Without this, the
# `from src import ...` line below raises ModuleNotFoundError. This is the
# single most common Windows + Jupyter stumble on a project laid out this way.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd

from src import config, socrata

DATASET_KEY = "service_requests"
dataset_id, label = config.DATASETS[DATASET_KEY]

print(label)
print("dataset id:", dataset_id)
print("source    :", socrata._resource_url(dataset_id))

## 1. What columns exist?

This is the first live API call. If it fails, the problem is network, throttling, or a changed dataset id, and nothing downstream is worth attempting until it succeeds.

In [ ]:
columns = socrata.get_columns(dataset_id)

print(f"{len(columns)} columns\n")
for i, column in enumerate(columns, start=1):
    print(f"{i:>3}. {column}")

## 2. What does a real row look like?

Column names lie. A field called `status` might hold three values or thirty. A date field might be `2024-03-01T00:00:00.000` or `03/01/2024`. Look at actual values before writing any filter.

In [ ]:
sample = socrata.get_sample(dataset_id, n=10)

with pd.option_context("display.max_columns", None, "display.width", 250):
    display(sample)

In [ ]:
# Transposed view: easier to read when there are many columns.
with pd.option_context("display.max_rows", None):
    display(sample.head(3).T)

## 3. Which columns are dates?

A service request usually carries several: when it was created, when it was last updated, when it was closed, and when it was due under the service-level commitment.

**The one that matters for step 1b is the created date.** Filtering the snapshot window on the closed date instead would silently drop every request that is still open, which is exactly the backlog this project measures.

In [ ]:
date_like = [c for c in columns if "date" in c.lower() or "time" in c.lower()]

print("Date-like columns:")
for column in date_like:
    print(" ", column)

print("\nSample values:")
if date_like:
    display(sample[date_like])

## 4. How big is the window?

Ask the server for a count before downloading anything. A `count(1)` request returns one number and costs nothing, and it tells you whether the snapshot will be 80,000 rows or 2,000,000.

Set `CREATED_DATE_FIELD` in `src/config.py` to the confirmed column name, restart the kernel, then run this cell.

In [ ]:
if config.CREATED_DATE_FIELD is None:
    print("CREATED_DATE_FIELD is not set yet. Set it in src/config.py, restart the kernel, rerun.")
else:
    field = config.CREATED_DATE_FIELD
    where = (
        f"{field} >= '{config.WINDOW_START}T00:00:00.000' "
        f"AND {field} <= '{config.WINDOW_END}T23:59:59.999'"
    )
    response = socrata._get(
        socrata._resource_url(dataset_id),
        {"$select": "count(1) AS n", "$where": where},
    )
    print("filter:", where)
    print("rows  :", response.text.strip())

## Next

Once the created-date column is confirmed and the row count looks manageable, run the snapshot from the Anaconda Prompt at the project root:

```
python -m src.ingest
```

Run it as a script, not from this notebook. The snapshot and its manifest are the project's provenance record, and they should come from a command anyone can rerun.